# Install Dependencies

In [ ]:
!pip install -q -U accelerate=='0.25.0' peft=='0.7.1' bitsandbytes=='0.41.3.post2' transformers=='4.36.1' trl=='0.7.4

In [ ]:
!pip install -q bitsandbytes
!pip install -q datasets
!pip install -q peft
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 3.3 MB/s eta 0:00:00


# login

In [ ]:
import huggingface_hub
huggingface_hub.login(Hugging_Face_TOKEN)

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import random
import pandas as pd
import re
import string
import sys
import argparse
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
from sklearn.utils import shuffle
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import emoji
import pyarabic.araby as araby

# Define Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['Positive', 'Negative', 'Neutral']
    mapping = {'Positive': 0, 'Negative': 1, 'Neutral':2, 'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred, digits = 4)
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2, 3])
    print('\nConfusion Matrix:')
    print(conf_matrix)

# Define Predict Function

In [ ]:
def predict(X_test, model, tokenizer):
    y_pred = []
    max_length = tokenizer.model_max_length
    for i in tqdm(range(len(X_test))):
        prompt = X_test.iloc[i]["text"]
        prompt = prompt[:max_length]
        result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
        answer = result[0]['generated_text'].split("=")[-1].lower()
        if "positive" in answer:
            y_pred.append("Positive")
        elif "negative" in answer:
            y_pred.append("Negative")
        elif "neutral" in answer:
            y_pred.append("Neutral")
        else:
            y_pred.append("none")
    return y_pred

# Load the Data

In [ ]:
data = pd.read_excel('All Climate Change Data - All Related.xlsx')

data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True, subset='text')
data.reset_index(drop=True, inplace = True)

data.shape

(23957, 4)

In [ ]:
data.head()

,location,date,text,sentiment
0,NaN,"2008, 10, 27, 17, 41, 17, tzinfo=datetime.time...",هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative
1,Türkiye,"2023, 2, 20, 19, 2, 5, tzinfo=datetime.timezon...",#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Neutral
2,NaN,"2023, 2, 17, 9, 45, tzinfo=datetime.timezone.utc",RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Neutral
3,NaN,"2024, 8, 17, 16, 1, 17, tzinfo=datetime.timezo...",رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Positive
4,NaN,"2024, 8, 11, 10, 32, 31, tzinfo=datetime.timez...",قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


# Preprocessing

In [ ]:
data['text'] = data['text'].str.replace(r'[^\w\s]+', '')
data['text'] = data['text'].str.replace("\s+", " ", regex=True)

data.head()

,location,date,text,sentiment
0,NaN,"2008, 10, 27, 17, 41, 17, tzinfo=datetime.time...",هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative
1,Türkiye,"2023, 2, 20, 19, 2, 5, tzinfo=datetime.timezon...",#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Neutral
2,NaN,"2023, 2, 17, 9, 45, tzinfo=datetime.timezone.utc",RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Neutral
3,NaN,"2024, 8, 17, 16, 1, 17, tzinfo=datetime.timezo...",رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Positive
4,NaN,"2024, 8, 11, 10, 32, 31, tzinfo=datetime.timez...",قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
data = data[['text', 'sentiment']]

In [ ]:
data['sentiment'].value_counts()

,count
sentiment,
Neutral,10307
Negative,8140
Positive,5510


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
data['text'] = data['text'].apply(normalize_arabic)

data.head()

,text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative
1,#عاجل | ادارة الكوارث والطوارئ التركية: ازالة ...,Neutral
2,RT @USUN: عُقد في مالطا هذا الاسبوع اول اجتماع...,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس الاهبة وب...,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Fix elongated words
    pattern = re.compile(r'(.)\1{2,}')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=r'\1'))
    # Normalize letters
    pattern = re.compile("[إأآا]")
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl="ا"))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
data['text'] = data_cleaning(data['text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

data['text'] = data['text'].apply(remove_ids)
data.head()

,text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Neutral
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس الاهبة وب...,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
data.isnull().sum()

,0
text,0
sentiment,0


In [ ]:
data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True)
data.shape

In [ ]:
data = data.rename(columns={'sentiment':'Final Label'})

# Split Data

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate.csv')

# Clean and convert to integer index arrays
test = data.loc[indecies['test'].dropna().astype(int).values]
train = data.loc[indecies['train'].dropna().astype(int).values]
val = data.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train = train.reset_index(drop=True)

def generate_prompt(data_point):
    return f"""
    You are an AI assistant specialized in climate change sentiment analysis.
    Classify the sentiment of the following Arabic sentence between square brackets to the left based on its emotional tone regarding climate change. Choose only one sentiment between: Positive, Negative, or Neutral for this Arabic sentence.

    Sentence: [{data_point["text"]}] = Sentiment: [{data_point["Final Label"]}]
            """.strip()

def generate_test_prompt(data_point):
    return f"""
    You are an AI assistant specialized in climate change sentiment analysis.
    Classify the sentiment of the following Arabic sentence between square brackets to the left based on its emotional tone regarding climate change. Choose only one sentiment between: Positive, Negative, or Neutral for this Arabic sentence.

    Sentence: [{data_point["text"]}] = Sentiment:
            """.strip()


train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["text"])
val = pd.DataFrame(val.apply(generate_prompt, axis=1),
                      columns=["text"])

In [ ]:
y_true = test["Final Label"]
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["text"])

train_data = Dataset.from_pandas(train)
eval_data = Dataset.from_pandas(train)

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Predict Without Fine-tuning

In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

# Fine-tune Model

In [ ]:
from trl import SFTConfig

peft_config = LoraConfig(
    lora_alpha=128,
    lora_dropout=0.05,
    r=64,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = SFTConfig(
    output_dir="logs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1, # 4
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0,
    fp16=False,
    bf16=False,
    max_grad_norm=1,
    max_steps=-1,
    warmup_ratio=0.01,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    eval_strategy="epoch",
    dataset_text_field="text",
    max_seq_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

Adding EOS to train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained("/content/drive/MyDrive/Fanar")

It is strongly recommended to train Gemma2 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Epoch,Training Loss,Validation Loss
1,2.159600,1.986088
2,1.895900,1.764272
3,1.543800,1.540034


# Predict

In [ ]:
from peft import PeftModel, PeftConfig
# Load the Lora model
model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/Fanar')

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=1,
                temperature=0.2
               )

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

100%|██████████| 3594/3594 [13:57<00:00,  4.29it/s]

Accuracy: 0.668

f1_score:  0.6606376784823857

Classification Report:
              precision    recall  f1-score   support

           0     0.7818    0.5719    0.6606       827
           1     0.6306    0.8976    0.7408      1221
           2     0.6702    0.5388    0.5973      1546
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.6683      3594
   macro avg     0.5206    0.5021    0.4997      3594
weighted avg     0.6824    0.6683    0.6606      3594


Confusion Matrix:
[[ 473   59  295    0]
 [   8 1096  115    2]
 [ 124  583  833    6]
 [   0    0    0    0]]


# Pred on Asad

In [ ]:
asad2 = pd.read_csv('train_all.csv')

asad2['Text'] = asad2['Text'].str.replace(r'[^\w\s]+', '')
asad2['Text'] = asad2['Text'].str.replace("\s+", " ", regex=True)

asad2['Text'] = asad2['Text'].apply(normalize_arabic)

asad2['Text'] = data_cleaning(asad2['Text'])
asad2['Text'] = asad2['Text'].apply(remove_ids)

asad2 = asad2.rename(columns={'Text': 'text'})
asad2.head()

,Tweet_id,sentiment,text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...
1,1221884400377499651,neutral,@ @ ليس حبا في ايران بقدر ماهو نكايه بترامب وحزبه
2,1221881406168731649,neutral,@ ابي اعرف الحاكم العربي المسلم اشلون ينام ماي...
3,1221882047691657217,neutral,@ @ في الخطاب تبع سليم سعاده حطت عالتويتر شو ق...
4,1221880371773673474,neutral,@ مفيش الكلام ده في الزمن


In [ ]:
train, asad20 = train_test_split(asad2,
                               test_size=20000,
                               stratify=asad2['sentiment'],
                               random_state=42)

asad20_test = pd.DataFrame(asad20.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pred_asad20 = predict(asad20_test, pipe, tokenizer)
evaluate(asad20['sentiment'].map(str.title), pred_asad20)

100%|██████████| 20000/20000 [1:17:57<00:00,  4.28it/s]

Accuracy: 0.627

f1_score:  0.6104040523348689

Classification Report:
              precision    recall  f1-score   support

           0     0.6987    0.1019    0.1779      3208
           1     0.3684    0.7764    0.4997      3207
           2     0.7626    0.7162    0.7387     13585
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.6273     20000
   macro avg     0.4574    0.3986    0.3541     20000
weighted avg     0.6892    0.6273    0.6104     20000


Confusion Matrix:
[[ 327  553 2325    3]
 [  10 2490  703    4]
 [ 131 3716 9729    9]
 [   0    0    0    0]]


# ASTD

In [ ]:
import pandas as pd

df = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])
df.head()

,text,Label
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,NEUTRAL


In [ ]:
df = df[df['Label']!='OBJ']
df['Label'] = df['Label'].map({
    'NEG': 'Negative',
    'NEUTRAL': 'Neutral',
    'POS': 'Positive'
})

df.head()

,text,Label
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,Positive
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,Negative
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,Neutral
5,#انتخبوا_العرص #انتخبوا_البرص #مرسى_رئيسى #اين...,Neutral
6,امير عيد هو اللي فعلا يتقال عليه ستريكر صريح #...,Positive


In [ ]:
df['text'] = df['text'].str.replace(r'[^\w\s]+', '')
df['text'] = df['text'].str.replace("\s+", " ", regex=True)

df['text'] = df['text'].apply(normalize_arabic)

df['text'] = data_cleaning(df['text'])
df['text'] = df['text'].apply(remove_ids)

df.dropna(inplace = True)
df = df.drop_duplicates(subset = 'text')

astd_test = pd.DataFrame(df.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pred_astd = predict(astd_test, pipe, tokenizer)
evaluate(df['Label'], pred_astd)

100%|██████████| 3223/3223 [12:01<00:00,  4.47it/s]

Accuracy: 0.516

f1_score:  0.49877924502257626

Classification Report:
              precision    recall  f1-score   support

           0     0.8471    0.0927    0.1671       777
           1     0.7354    0.7148    0.7250      1641
           2     0.2730    0.5193    0.3579       805
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.5160      3223
   macro avg     0.4639    0.3317    0.3125      3223
weighted avg     0.6468    0.5160    0.4988      3223


Confusion Matrix:
[[  72   47  654    4]
 [   4 1173  459    5]
 [   9  375  418    3]
 [   0    0    0    0]]


# SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'Negative',
    'neutral': 'Neutral',
    'positive': 'Positive'
})
semeval.head()

,text,sentiment
0,إجبار أبل على التعاون على فك شفرة اجهزتها http...,Positive
1,RT @20fourMedia: #غوغل تتحدى أبل وأمازون بأجهز...,Positive
2,جوجل تنافس أبل وسامسونج بهاتف جديد https://t.c...,Positive
3,رئيس شركة أبل: الواقع المعزز سيصبح أهم من الطع...,Positive
4,ساعة أبل في الأسواق مرة أخرى https://t.co/dY2x...,Neutral


In [ ]:
semeval['text'] = semeval['text'].str.replace(r'[^\w\s]+', '')
semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)

semeval['text'] = semeval['text'].apply(normalize_arabic)

semeval['text'] = data_cleaning(semeval['text'])
semeval['text'] = semeval['text'].apply(remove_ids)

semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')

semeval_test = pd.DataFrame(semeval.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pred_semeval = predict(semeval_test, pipe, tokenizer)
evaluate(semeval['sentiment'], pred_semeval)

100%|██████████| 3287/3287 [12:55<00:00,  4.24it/s]

Accuracy: 0.549

f1_score:  0.49120161111443106

Classification Report:
              precision    recall  f1-score   support

           0     0.6667    0.0055    0.0108       733
           1     0.7551    0.5018    0.6029      1100
           2     0.5213    0.8590    0.6488      1454
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.5491      3287
   macro avg     0.4858    0.3416    0.3157      3287
weighted avg     0.6320    0.5491    0.4912      3287


Confusion Matrix:
[[   4   30  688   11]
 [   0  552  459   89]
 [   2  149 1249   54]
 [   0    0    0    0]]
